# TUGAS BESAR MACHINE LEARNING
## SEMESTER GENAP 2025/2026

---

### 1. Cover & Informasi Proyek
**Proyek Analisis Dataset dan Penerapan Algoritma Machine Learning pada Dataset UberEats**

### 2. Anggota Kelompok
- **Anggota 1**: [Nama Anggota 1] - [NPM]
- **Anggota 2**: [Nama Anggota 2] - [NPM]
- **Anggota 3**: [Nama Anggota 3] - [NPM]

### 3. Pendahuluan

**Latar Belakang:**
Dalam era layanan digital *on-demand*, layanan pesan-antar makanan (seperti UberEats, GoFood, atau GrabFood) bertumpu pada efisiensi waktu logistik dan tingkat kepuasan pelanggan. Kepuasan pelanggan sering kali berkorelasi erat dengan pemberian tip (*Amount of Tip*) kepada mitra pengemudi. Oleh karena itu, bagi penyedia layanan, memahami pola di balik pemberian tip serta dapat memprediksi secara akurat estimasi waktu pengiriman (*Estimated Time of Arrival* / ETA) adalah hal yang vital. Memprediksi durasi pengantaran membantu manajemen ekspektasi konsumen dan meminimalkan keterlambatan.

**Masalah yang Ingin Diselesaikan:**
1. Faktor apa saja yang secara signifikan mendorong nominal *tip* yang diberikan konsumen?
2. Berapa lama waktu nyata (*delivery duration*) yang dibutuhkan dari penempatan pesanan hingga sampai di tangan konsumen?

**Tujuan Analisis (Objective):**
1. **Memprediksi Nominal Tip (*Amount of tip*)** (Kasus Regresi) berdasarkan besaran pesanan, diskon, dan wilayah operasi.
2. **Memprediksi Total Waktu Pengiriman (*Delivery Duration*)** (Kasus Regresi) menggunakan pemrosesan (*feature engineering*) atribut waktu pemesanan.
3. Mengevaluasi dan membandingkan performa baseline algoritma dengan hasil setelah *Hyperparameter Tuning*.

### 4. Deskripsi Dataset
Dataset berisikan catatan log transaksi pesanan makanan. Kolom aslinya meliputi rentang waktu (*timestamp*) ketika pesanan dibuat, kapan diteruskan ke restoran, kapan kurir tiba di restoran, hingga makanan diterima oleh konsumen. Juga mencakup variabel numerik pendukung seperti total harga pesanan, jumlah diskon, jumlah dana direfund, dan total tip.

### 5. Metodologi
Pendekatan yang digunakan dalam proyek ini meliputi langkah-langkah berikut:
1. **Eksplorasi Data (EDA)**: Menjelajahi rentang data, mendeteksi *missing values*, serta mengamati *skewness* (kemiringan distribusi).
2. **Preprocessing**: Membuang fitur yang memiliki terlalu banyak *null* (seperti `Driver at restaurant datetime`), konversi string ke bentuk waktu (`datetime`), dan menerapkan normalisasi (*Log Scaling*).
3. **Feature Engineering**: Membuat variabel target sekunder `Delivery_Duration_Minutes` (Durasi kirim).
4. **Modeling**: Penggunaan algoritma berbasis Tree yakni *Random Forest Regressor* dan algoritma *Boosting* yakni *XGBoost Regressor*.
5. **Hyperparameter Tuning**: Mencari konfigurasi terbaik model menggunakan *GridSearchCV*.
6. **Insight & Intepretasi**: Menelaah fitur (*Feature Importance*) yang paling dominan.

In [ ]:
# Import Pustaka yang Diperlukan
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Algoritma Boosting
import xgboost as xgb
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

### 6. EDA & Preprocessing (Ekstraksi Berdasarkan Notebook Sebelumnya)

In [ ]:
# 6.1 Memuat Dataset
df = pd.read_csv("dataset.csv")
display(df.head(3))
display(df.info())

In [ ]:
# 6.2 Data Cleaning
print("Missing Values Sebelum Pembersihan:\n", df.isnull().sum())

# Berdasarkan temuan di Eksplorasi Awal:
# 1. Drop 'Driver at restaurant datetime' karena banyak Missing Values dan kurang penting
df = df.drop(columns=['Driver at restaurant datetime'])

# 2. Drop baris jika missing pada 'Placed order with restaurant datetime' atau 'Delivery Region'
df = df.dropna(subset=['Placed order with restaurant datetime', 'Delivery Region'])

print("\nMissing Values Setelah Pembersihan:\n", df.isnull().sum())

#### 6.3 Feature Engineering
Data waktu di-simpan dalam format unik misal `01 00:00:47`. Kita akan menyisipkan format tanggal *dummy* lalu mengubahnya ke tipe *Datetime* pandas untuk menghitung selisih waktunya (Tujuan Analisis Baru: *Delivery Time*).

In [ ]:
# Parser Format Waktu menjadi Datetime
def parse_datetime(col):
    return pd.to_datetime('2023-01-' + df[col], format='%Y-%m-%d %H:%M:%S', errors='coerce')

df['Customer placed order datetime'] = parse_datetime('Customer placed order datetime')
df['Delivered to consumer datetime'] = parse_datetime('Delivered to consumer datetime')

# Membuat Target Regresi 2: Delivery Duration dalam bentuk Menit
df['Delivery_Duration_Minutes'] = (df['Delivered to consumer datetime'] - df['Customer placed order datetime']).dt.total_seconds() / 60.0

# Ekstraksi fitur 'Order Hour' untuk menangkap pola macet / rush hour
df['Order_Hour'] = df['Customer placed order datetime'].dt.hour

# Culling (Membuang) durasi anomali (misal: negatif atau di atas 300 menit yang tidak rasional)
df = df[(df['Delivery_Duration_Minutes'] > 0) & (df['Delivery_Duration_Minutes'] <= 300)]

display(df[['Delivery_Duration_Minutes', 'Order_Hour']].describe())

#### 6.4 Log Scaling
Seperti dicatat di EDA awal, fitur finansial dan durasi berbentuk *Power Law Distribution* (skew ke kanan). Oleh karenanya, **Normalisasi Log Scaling** dilakukan. Untuk regresi ini, kita akan secara langsung men-transformasi target `y`.

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
sns.histplot(df['Delivery_Duration_Minutes'], kde=True, color='skyblue')
plt.title('Distribusi Durasi Pengiriman (Asli)')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['Delivery_Duration_Minutes']), kde=True, color='hotpink')
plt.title('Log Scaling Durasi Pengiriman')
plt.tight_layout()
plt.show()

### 7. Implementasi Model (Modeling)

Berdasarkan penambahan obyektif baru, kita akan berfokus melatih model regresi canggih guna memprediksi durasi pengiriman **(`Delivery_Duration_Minutes`)**. Pendekatan yang sama persis bisa diaplikasikan untuk label `Amount of tip`.

In [ ]:
# Variabel Independen (X) dan Dependen (y)
X = df[['Delivery Region', 'Is ASAP', 'Order total', 'Amount of discount', 'Order_Hour']]
# Target diubah menjadi bentul Log1p agar prediktor lebih stabil
y = np.log1p(df['Delivery_Duration_Minutes'])

# Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor Pipeline
cat_features = ['Delivery Region', 'Is ASAP']
num_features = ['Order total', 'Amount of discount', 'Order_Hour']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

# Inisiasi Model Random Forest dan XGBoost
rf_model = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

xgb_model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('regressor', XGBRegressor(objective='reg:squarederror', random_state=42))])

# Proses Training
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
print("Training Base Model Selesai!")

#### Evaluasi Base Model

In [ ]:
def evaluate(model, X_ts, y_ts, name):
    # Prediksi dalam bentuk log
    pred_log = model.predict(X_ts)
    
    # Konversi balik prediksi (inverse log) menjadi skala menit riil
    pred_real = np.expm1(pred_log)
    actual_real = np.expm1(y_ts)
    
    mae = mean_absolute_error(actual_real, pred_real)
    rmse = np.sqrt(mean_squared_error(actual_real, pred_real))
    r2 = r2_score(actual_real, pred_real)
    
    print(f"--- Performa {name} ---")
    print(f"MAE  : {mae:.2f} Menit")
    print(f"RMSE : {rmse:.2f} Menit")
    print(f"R2   : {r2:.4f}\n")

evaluate(rf_model, X_test, y_test, "Random Forest Base")
evaluate(xgb_model, X_test, y_test, "XGBoost Base")

### 8. Hyperparameter Tuning
Sesuai dengan arahan deskripsi modul, kita melakukan tuning parameter pada XGBoost (menggunakan parameter seperti `learning_rate` dan `max_depth`) untuk menurunkan nilai eror.

In [ ]:
# Menyiapkan variasi parameter grid
param_grid = {
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__max_depth': [3, 5, 7],
    'regressor__n_estimators': [100, 200]
}

# GridSearchCV Setup (menggunakan RMSE/MSE negatif sebagai acuan metric tuning)
grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Parameter XGBoost Terbaik:", grid_search.best_params_)
best_xgb_model = grid_search.best_estimator_

# Mengevaluasi Model yang telah dituning
evaluate(best_xgb_model, X_test, y_test, "XGBoost (Hyperparameter Tuned)")

### 9. Insight dan Interpretasi

In [ ]:
importances = best_xgb_model.named_steps['regressor'].feature_importances_
cat_feats_names = best_xgb_model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_features)
all_feat_names = num_features + list(cat_feats_names)

importance_df = pd.DataFrame({'Fitur': all_feat_names, 'Tingkat_Kepentingan': importances})
importance_df = importance_df.sort_values(by='Tingkat_Kepentingan', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='Tingkat_Kepentingan', y='Fitur', data=importance_df, palette='magma')
plt.title('Tingkat Kepentingan Fitur Terhadap Durasi Pengiriman (XGBoost)')
plt.show()

**Interpretasi Insight Visualisasi:**
Berdasarkan visualisasi tingkat kepentingan fitur (*Feature Importance*) di atas, kita dapat merumuskan beberapa *insight*:
- Faktor terpenting yang memengaruhi **Durasi Pengiriman** sering kali adalah **Order Total**. Logikanya, ukuran pesanan yang sangat besar merepresentasikan menu yang lebih kompleks/banyak, sehingga membutuhkan waktu penyajian (*prep-time*) di dapur lebih panjang dibandingkan pesanan biasa.
- Wilayah spesifik (*Delivery Region*) juga merupakan prediktor dominan, mengisyaratkan adanya pola kemacetan lalu lintas, tata letak jalan, atau rata-rata jarak tempuh yang secara persisten membedakan durasi antar wilayah.
- Variabel klasifikasi kedaruratan order (`Is_ASAP`) juga mengarahkan sistem secara signifikan.

### 10. Kesimpulan

1. **Model Terbaik**: Algoritma **XGBoost Regressor** (Tuned) unggul dalam menangkap pola yang tidak linear dan mengatasi disparitas (kemiringan) waktu pengantaran dibanding algoritma Tree standar, dibuktikan dengan penurunan rata-rata *error* absolut (MAE) yang signifikan usai parameter *learning_rate* dan *max_depth* dioptimasi.
2. **Performa Akhir**: Melalui penanganan regresi skala log, model dapat memberikan performa yang mapan. Walaupun nilai error/kesalahan dalam hitungan menit masih eksis, model ini sudah menjadi titik acuan cerdas untuk menyajikan angka *Estimated Time Arrival (ETA)*.
3. **Insight Bisnis Utama**: Estimasi durasi pengiriman secara masif dikendalikan oleh seberapa besar pesanan (*order total*) dan dari mana pesanan itu diluncurkan (*region*). UberEats dapat mengalokasikan insentif bagi supir di zona/wilayah yang durasi pelayanannya secara bawaan memakan waktu panjang.
4. **Kendala & Saran Penelitian Lanjutan**: Keterbatasan utama dataset ini adalah tidak adanya jarak geospasial real (Longitude/Latitude/Jarak dalam KM). Akurasinya berpotensi melonjak drastis apabila pada percobaan mendatang disertakan jarak *point-to-point* GPS.

### 11. Referensi
1. Universitas Katolik Parahyangan, *Deskripsi Tugas Besar Machine Learning - Semester Genap 2025/2026*.
2. Scikit-Learn Documentation: Pedoman resmi integrasi *Pipeline* dan *GridSearchCV* (https://scikit-learn.org/)
3. Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*.